# Credit rating

## Table of contents

[1 Data loading](#1-data-loading)

[2 Data completeness assessment](#2-data-completeness-assessment)

[3. Replacing missing values](#3-replacing-missing-values)

[4. Type conversion](#4-type-conversion)

[5. Handling duplicates](#5-handling-duplicates)

[6. Data categorization ](#6-data-categorization)

[7. Hypothesis testing](#7-hypothesis-testing)

[8. Conclusion](#8-conclusion)

## 1. Data loading

We import the modules and load the dataset. We get acquainted with the data using the first 20 rows of the dataset as an example.

In [1]:
import pandas as pd
from matplotlib import pyplot as plt

try:
    data = pd.read_csv(r"C:\Users\yka\Documents\Practicum\DS_Practicum\Credit_Scoring\source\data.csv")
except:
    data = pd.read_csv("/datasets/data.csv")

data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


We get acquainted with the dataset's metadata in order to convert it to the correct data types.

In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


# 2. Data completeness assessment

In [3]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

There are missing values in the data. It is suspicious that the missing values are only in two columns and that their number is the same.

In [4]:
len(data[(data["days_employed"].isna()) & (data["total_income"].isna())])

2174

People who have no information about their length of employment also have no information about their total income. Let us take a closer look at the correlations between the rows in which the information about length of employment and income is missing.

In [5]:
data[data["days_employed"].isna()].corr()

,children,days_employed,dob_years,education_id,family_status_id,debt,total_income
children,1.000000,NaN,-0.152431,-0.033552,-0.036475,-0.005732,NaN
days_employed,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dob_years,-0.152431,NaN,1.000000,0.091549,-0.066621,-0.052709,NaN
education_id,-0.033552,NaN,0.091549,1.000000,-0.016376,0.054495,NaN
family_status_id,-0.036475,NaN,-0.066621,-0.016376,1.000000,0.005102,NaN
debt,-0.005732,NaN,-0.052709,0.054495,0.005102,1.000000,NaN
total_income,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Checking the relationship between the columns with missing values in the `days_employed` and `total_income` columns showed that the correlation of the missing values in this column with the other data attributes is low. Consequently, we can conclude that the missing values occurred at random.

# 3. Replacing missing values

We will replace the missing values in `total_income` with the median values of the corresponding employment types.

In [6]:
median_income_by_type = data.groupby("income_type")["total_income"].median()

for type_key in median_income_by_type.keys():
    data.loc[(data["total_income"].isna()) & (data["income_type"] == type_key), "total_income"] = median_income_by_type[type_key]
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income           0
purpose                0
dtype: int64

We take the absolute values of the data in `days_employed`.

In [7]:
data["days_employed"] = abs(data["days_employed"])
data.head()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу


For each employment type, let us look at the median value of the length of employment `days_employed` in days.

In [8]:
days_employed_by_type = data.groupby("income_type")["days_employed"].median()
days_employed_by_type

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

Let us examine the list of unique values of the `children` column.

In [9]:
data["children"].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

Let us remove the anomalous values -1 and 20 in the children column (we will consider these outliers).

In [10]:
data = data.drop(data[(data["children"] == -1) | (data["children"] == 20)].index)
data["children"].unique()

array([1, 0, 3, 2, 4, 5])

Since the missing values in the `days_employed` column are random, let us fill the missing values in the `days_employed` column with the median values for each employment type `income_type` in order to preserve more typical values for each of the borrower classes.

In [11]:
days_employed_by_type = data.groupby("income_type")["days_employed"].median()

for income_type in days_employed_by_type.keys():
    data.loc[(data["days_employed"].isna()) & (data["income_type"] == income_type), "days_employed"] = days_employed_by_type[income_type]

data["days_employed"].isna().sum()

0

# 4. Type conversion

Let us replace the floating-point data type in the `total_income` column with an integer one using the `astype()` method.

In [12]:
data["total_income"] = data["total_income"].astype("int")
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21402 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21402 non-null  int64  
 1   days_employed     21402 non-null  float64
 2   dob_years         21402 non-null  int64  
 3   education         21402 non-null  object 
 4   education_id      21402 non-null  int64  
 5   family_status     21402 non-null  object 
 6   family_status_id  21402 non-null  int64  
 7   gender            21402 non-null  object 
 8   income_type       21402 non-null  object 
 9   debt              21402 non-null  int64  
 10  total_income      21402 non-null  int64  
 11  purpose           21402 non-null  object 
dtypes: float64(1), int64(6), object(5)
memory usage: 2.1+ MB


# 5. Handling duplicates 

Let us replace the implicit duplicates in the `education` column by converting them to lower case.

In [13]:
data["education"] = data["education"].str.lower()

Let us count the explicit duplicates.

In [14]:
data.duplicated().sum()

71

Let us remove the explicit duplicates.

In [15]:
data = data.drop_duplicates()
data.duplicated().sum()

0

# 6. Data categorization 

Let us classify borrowers by income level:

- 0–30000 — 'E'
- 30001–50000 — 'D'
- 50001–200000 — 'C'
- 200001–1000000 — 'B'
- 1000001 and above — 'A'

In [16]:
def categorize_income(income):
    if income <= 30000:
        return 'E'
    elif income > 30000 and income <= 50000:
        return 'D'
    elif income > 50000 and income <= 200000:
        return 'C'
    elif income > 200000 and income <= 1000000:
        return 'B'
    else:
        return 'A'

data["total_income_category"] = data["total_income"].apply(categorize_income)
data["total_income_category"]

0        B
1        C
2        C
3        B
4        C
        ..
21520    B
21521    C
21522    C
21523    B
21524    C
Name: total_income_category, Length: 21331, dtype: object

Let us look at the list of unique purposes for taking a loan from the `purpose` column.

In [17]:
data["purpose"].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

Let us create a function that, based on the data from the `purpose` column, will form a new column `purpose_category`, which will include the following categories:

- 'car transactions'
- 'real estate transactions'
- 'holding a wedding'
- 'obtaining an education'

The source values of `purpose` are stored in Russian, so the function matches Russian substrings, while the category labels it writes are in English.

In [18]:
def categorize_purpose(purpose):
    # The substrings below are matched against the raw Russian values of the
    # `purpose` column, so they are kept in the source language on purpose.
    if "авто" in purpose:
        return "car transactions"
    elif "жил" in purpose or "недвиж" in purpose:
        return "real estate transactions"
    elif "свадьб" in purpose:
        return "holding a wedding"
    elif "образов" in purpose:
        return "obtaining an education"
    else:
        return None

data["purpose_category"] = data["purpose"].astype("string").apply(categorize_purpose)

# 7. Hypothesis testing

Let us formulate the questions corresponding to the hypotheses being tested:
- Is there a relationship between the number of children `children` and repaying the loan `debt` on time?
- Is there a relationship between marital status `family_status_id` and repaying the loan `debt` on time?
- Is there a relationship between the income level `total_income_category` and repaying the loan `debt` on time?
- How do the different loan purposes `purpose_category` affect its repayment `debt` on time?

To draw a conclusion about the influence of the number of children on debt delinquency, let us calculate the mean values for borrowers with different numbers of children.

In [19]:
pd.pivot_table(data, values="debt", index="children", aggfunc=["count", "mean"])

,count,mean
,debt,debt
children,,
0,14091,0.075438
1,4808,0.092346
2,2052,0.094542
3,330,0.081818
4,41,0.097561
5,9,0.000000


Conclusion: the **more children** a borrower has, the more he is **prone to delinquency**. The deviations from this logic for the values 3 and 5 are probably due to the smaller sample of such borrowers. The conclusion can be explained by the large number of unplanned expenses on children who are dependants of the respective borrowers.

In [20]:
100 - (data[data["children"] == 0]["debt"].mean() / data[data["children"] != 0]["debt"].mean()) * 100

18.359830438048476

The probability of debt repayment by a borrower with children is 18% lower than by a borrower without children.

To draw a conclusion about the influence of marital status on debt delinquency, let us calculate the mean values for borrowers with different marital statuses.

In [21]:
family_status = pd.pivot_table(data, values="debt", index="family_status", aggfunc=["count", "mean"])
family_status

,count,mean
,debt,debt
family_status,,
Не женат / не замужем,2796,0.097639
в разводе,1189,0.070648
вдовец / вдова,951,0.066246
гражданский брак,4134,0.093130
женат / замужем,12261,0.075606


In [22]:
(family_status["mean"]["debt"].max() - family_status["mean"]["debt"].min()) * 100

3.1393428196206385

We can conclude that different categories of borrowers by marital status have a different probability of debt repayment; however, **the influence of marital status is insignificant** since the largest difference between the categories reaches a little more than 3%.

It can also be noticed that the categories of borrowers who were in a relationship and for some reason ceased to be in one are slightly more solvent. There is not enough information to analyse the reasons for this observation.

To draw a conclusion about the influence of the income class on debt delinquency, let us calculate the mean values for borrowers with different income classes.

In [23]:
pd.pivot_table(data, values="debt", index="total_income_category", aggfunc=["count", "mean"])

,count,mean
,debt,debt
total_income_category,,
A,25,0.080000
B,5014,0.070602
C,15921,0.084982
D,349,0.060172
E,22,0.090909


In [24]:
data["debt"].mean()

0.08119638085415593

Comparing the solvency of borrowers with different income classes with the average solvency of borrowers allows us to conclude that borrowers with **income class E are less solvent** than the rest. At the same time, **the most solvent are borrowers of class D, B**. 

Probably the information is not reliable enough since the samples for borrowers with classes A, D, E are significantly smaller than the rest.

To draw a conclusion about the influence of the loan purpose on debt delinquency, let us calculate the mean values for borrowers with different loan purposes.

In [25]:
pd.pivot_table(data, values="debt", index="purpose_category", aggfunc=["count", "mean"])

,count,mean
,debt,debt
purpose_category,,
car transactions,4279,0.093480
holding a wedding,2313,0.079118
obtaining an education,3988,0.092528
real estate transactions,10751,0.072551


In [26]:
(0.093480 - 0.072551) * 100

2.092899999999999

The difference between the solvency of clients by loan purpose is insignificant and fluctuates within 2%.

Payments on time are most likely for loans for real estate transactions.

Payments on time are least likely for loans for car transactions.

[Data completeness assessment](#2-data-completeness-assessment)

Since the number of missing values in the `days_employed` and `total_income` columns was the same, it was reasonable to check whether the data was missing in the same rows. The check showed that the missing values were in the same rows. The analysis of correlations for the rows with missing values showed that the missing values are random and do not depend on the other attributes of the dataset. 

It can be assumed that the information about length of employment and total income was requested from the corresponding government agencies and is absent for these borrowers or was requested incorrectly. It is worth clarifying more precise information with the creators of the dataset. 

The data in the quantitative variables was filled with the median values of the corresponding categories in order to preserve the peculiarities within the borrower categories. Since we do not know anything about the distribution of the data in the `days_employed` and `total_income` columns, let us fill the missing values with the median values in order to reduce the distortion of the averaging by outliers.

# 8. Conclusion

The most **significant differences** are in the factor of the presence and absence of children. Borrowers **without children are 18% more solvent** compared to borrowers with children.

Other categories influence the solvency of clients to a lesser extent.

**The most solvent** are borrowers **with the marital status "divorced" (в разводе) and "widower / widow" (вдовец / вдова)**.
**The least solvent** are borrowers **with the marital status "unmarried" (Не женат / не замужем) and those in a "common-law marriage" (гражданский брак)**.

It is noticeable that the categories of borrowers who **were in a relationship** and for some reason ceased to be in one are slightly **more solvent**.

Comparing the solvency of borrowers with different income classes with the average solvency of borrowers allows us to conclude that borrowers with **income class E are less solvent** than the rest. At the same time, **the most solvent are borrowers of class D, B**. 

The difference between the solvency of clients by loan purpose is insignificant and fluctuates within 2%.
**The most solvent** are borrowers **with the purpose of real estate transactions**.
**The least solvent** are borrowers **with the purpose of car transactions**.

In [27]:
df1 = pd.DataFrame({'a': [1, 2, 3, 4], 'b': ['A', 'B', 'C', 'D']})
df2 = pd.DataFrame({'a': [2, 2, 2, 2], 'c': ['E', 'F', 'G', 'H']})
print(df1)
print()
print(df2)
print()
print (df1.join(df2, on='a', rsuffix='_y')['c']) 

   a  b
0  1  A
1  2  B
2  3  C
3  4  D

   a  c
0  2  E
1  2  F
2  2  G
3  2  H

0      F
1      G
2      H
3    NaN
Name: c, dtype: object
